In [13]:
#교차검증 
#검증데이터(validation data)와 훈련/검증 반복 및 테스트 절차
import pandas as pd
import requests
from io import StringIO

#url링크 안전하게 접속하여 링크 받아오기
url= r'https://bit.ly/wine_csv_data'
r = requests.get(url, timeout=10)
r.raise_for_status()

#url링크 데이터 읽고 df로 만들기
wine = pd.read_csv(StringIO(r.text), low_memory=False)
wine.head()
wine.info()
wine.describe()

print(wine.columns)
#데이터 셋 분할
input = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']


from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(input, target)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   alcohol  6497 non-null   float64
 1   sugar    6497 non-null   float64
 2   pH       6497 non-null   float64
 3   class    6497 non-null   float64
dtypes: float64(4)
memory usage: 203.2 KB
Index(['alcohol', 'sugar', 'pH', 'class'], dtype='object')


In [14]:
#교차검증: 충분하고 안정적인 훈련과 검증하기
#k-fold 교차검증. k는 훈련 세트를 나누는 층. 일반적으로 5와 10사용
#cross_validate
from sklearn.model_selection import cross_validate, StratifiedKFold
import numpy as np
from sklearn.tree import DecisionTreeClassifier

#하이퍼파라미터 튜닝: 그리드서치 클래스 - 최적의 하이퍼 파라미터를 동시에 찾으면서 교차검증까지 수행
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': np.arange(0.0001, 0.0006, 0.0001)} #파라미터는 딕셔너리 형태로 전달. 설정파라미터가 여러개 일 때 리스트 안의 딕셔너리도 가능
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=1)
gs.fit(train_input, train_target)
dt = gs.best_estimator_
print(dt.score(train_input, train_target))
print(gs.best_params_)
print(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'])
print('\n', '\n', '\n')

params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001), 
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10)}
gs2 = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=1)
gs2.fit(train_input, train_target)
dt2 = gs2.best_estimator_
print(dt2.score(train_input, train_target))
print(gs2.best_params_)
print(np.max(gs2.cv_results_['mean_test_score']))


0.909688013136289
{'min_impurity_decrease': np.float64(0.00030000000000000003)}
[0.86330122 0.86884263 0.8696644  0.86843279 0.86863813]
[{'min_impurity_decrease': np.float64(0.0001)}, {'min_impurity_decrease': np.float64(0.0002)}, {'min_impurity_decrease': np.float64(0.00030000000000000003)}, {'min_impurity_decrease': np.float64(0.0004)}, {'min_impurity_decrease': np.float64(0.0005)}]

 
 

0.9070197044334976
{'max_depth': 12, 'min_impurity_decrease': np.float64(0.00030000000000000003), 'min_samples_split': 2}
0.8704853367029959


In [ ]:
#균등한 샘플을 랜덤하게 뽑기 (사이파이 stats 클래스 중 정수(randint), 실수(uniform))
from scipy.stats import uniform, randint
rgen = uniform(0, 1)
np.round(rgen.rvs(1000), decimals=3)
num, count = np.unique(rgen.rvs(1000), return_counts=True)
print(np.round(num, decimals=3), np.sum(count))
it = randint(0, 10)
it.rvs(100)
np.unique(it.rvs(1000), return_counts=True)

[0.001 0.001 0.002 0.002 0.003 0.004 0.007 0.007 0.01  0.011 0.011 0.012
 0.012 0.014 0.015 0.016 0.016 0.018 0.018 0.018 0.018 0.019 0.019 0.022
 0.024 0.026 0.026 0.031 0.031 0.032 0.037 0.037 0.038 0.039 0.04  0.04
 0.041 0.044 0.051 0.052 0.054 0.056 0.057 0.057 0.058 0.058 0.058 0.06
 0.062 0.065 0.065 0.066 0.066 0.066 0.066 0.068 0.068 0.069 0.069 0.07
 0.07  0.07  0.075 0.076 0.078 0.078 0.078 0.08  0.081 0.082 0.082 0.083
 0.084 0.084 0.084 0.085 0.085 0.087 0.088 0.089 0.091 0.091 0.092 0.092
 0.092 0.094 0.095 0.096 0.097 0.098 0.098 0.101 0.101 0.101 0.103 0.104
 0.104 0.105 0.107 0.107 0.107 0.108 0.109 0.109 0.111 0.112 0.113 0.115
 0.116 0.117 0.117 0.118 0.119 0.119 0.12  0.121 0.121 0.123 0.124 0.125
 0.125 0.126 0.127 0.128 0.129 0.13  0.131 0.131 0.132 0.135 0.135 0.137
 0.137 0.138 0.139 0.141 0.142 0.145 0.145 0.145 0.147 0.15  0.151 0.151
 0.152 0.152 0.153 0.155 0.156 0.156 0.157 0.158 0.159 0.159 0.16  0.162
 0.163 0.164 0.167 0.168 0.169 0.169 0.17  0.17  0.17 

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([115, 102, 100,  70, 101, 118,  92, 107,  90, 105]))

In [48]:
params = {'max_depth': randint(20, 50), 
          'min_impurity_decrease': uniform(0.0001, 0.001),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25)}

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=1, n_iter=100, cv=cv, random_state=42)
rs.fit(train_input, train_target)
dt = rs.best_estimator_
print(rs.best_params_)
print(np.max(rs.cv_results_['mean_test_score']))
print(dt.score(train_input, train_target))
print(dt.score(test_input, test_target))

{'max_depth': 31, 'min_impurity_decrease': np.float64(0.0003517822958253642), 'min_samples_leaf': 2, 'min_samples_split': 4}
0.8696649779513246
0.9002463054187192
0.8683076923076923


In [52]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint, uniform
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
params = {'max_depth': randint(20, 50).rvs(20), 
          'min_impurity_decrease': uniform(0.0001, 0.001).rvs(10), 
          'min_samples_split': randint(2, 25).rvs(10),
          'min_samples_leaf': randint(1, 25).rvs(10)}
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=1, cv=cv)
gs.fit(train_input, train_target)
dtc = gs.best_estimator_
print(gs.best_params_)
print(gs.cv_results['mean_test_score'])
print(dtc.score(train_input, train_target))
print(dtc.score(test_input, test_target))

KeyboardInterrupt: 

In [60]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

params2 = {'max_depth': randint(20, 50), 
          'min_impurity_decrease': uniform(0.0001, 0.001), 
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25)}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params2, n_jobs=1, n_iter=100, cv=cv, return_train_score=True, random_state=42)
rs.fit(train_input, train_target)
# dt2 = rs.best_estimator_
# print(dt.score(train_input, train_target))
# print(dt.score(test_input, test_target))
# print(rs.best_params_)
# print(rs.cv_results_['mean_test_score'].shape, np.max(rs.cv_results_['mean_test_score']))

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': <scipy.stats....00241CBE81490>, 'min_impurity_decrease': <scipy.stats....002418240E910>, 'min_samples_leaf': <scipy.stats....00241828319D0>, 'min_samples_split': <scipy.stats....00241822188D0>}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",100
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Gu

In [63]:
par = {'max_depth': randint(20, 50), 
          'min_impurity_decrease': uniform(0.0001, 0.001), 
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25)}

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
rsc = RandomizedSearchCV(DecisionTreeClassifier(splitter='best', random_state=42), par, n_iter=100, n_jobs=1, random_state=42)
rsc.fit(train_input, train_target)
dt1 = rsc.best_estimator_
print(f"best: {dt1.score(test_input, test_target)}")

rsc2 = RandomizedSearchCV(DecisionTreeClassifier(splitter='random', random_state=42), par, n_iter=100, n_jobs=1, random_state=42)
rsc2.fit(train_input, train_target)
dt2 = rsc2.best_estimator_
print(f"random: {dt2.score(test_input, test_target)}")



best: 0.8652307692307692
random: 0.8541538461538462
